# IPF baseline: weighted full seed

This notebook runs the short, library-backed IPF implementation using the same Census marginals, weighted seed, controlled attributes, and zone field as `IO/configs/direct_count_weighted_seed_full_two_stage.yml`.

The configured seed has no `hhinc = Negative income` records. Matching the configuration's `include_zero_cell_values: false` setting, that Census control is folded into `Nil income` before fitting. The notebook writes both fractional IPF weights and deterministic integerised counts, plus per-zone diagnostics.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import yaml
from ipfn import ipfn
from PopSynthesis.Generator_data.generate_combine_census.utils import TRS

CONFIG_RELATIVE = Path("IO/configs/direct_count_weighted_seed_full_two_stage.yml")
OUTPUT_RELATIVE = Path("IO/output/ipf_weighted_seed_full_two_stage_baseline")
MAX_ZONES = None  # Set to a small integer for a quick smoke test.
CONVERGENCE_RATE = 1e-5
MAX_ITERATIONS = 500
SAVE_FRACTIONAL_WEIGHTS = True
RANDOM_SEED = 42  # TRS uses NumPy's random generator.

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / CONFIG_RELATIVE).is_file():
            return candidate
    raise FileNotFoundError(f"Could not find {CONFIG_RELATIVE} above {start}")

repo_root = find_repo_root(Path.cwd())
config_path = repo_root / CONFIG_RELATIVE
output_dir = repo_root / OUTPUT_RELATIVE
output_dir.mkdir(parents=True, exist_ok=True)
repo_root, config_path, output_dir

(PosixPath('/Users/bobla/Documents/repos/PopSyn_Monash'),
 PosixPath('/Users/bobla/Documents/repos/PopSyn_Monash/IO/configs/direct_count_weighted_seed_full_two_stage.yml'),
 PosixPath('/Users/bobla/Documents/repos/PopSyn_Monash/IO/output/ipf_weighted_seed_full_two_stage_baseline'))

## Load the same configuration and input files

Paths are resolved relative to the YAML file, exactly as the project CLI resolves them.

In [2]:
with config_path.open() as stream:
    runs = yaml.safe_load(stream)

if len(runs) != 1:
    raise ValueError(f"Expected one run in {config_path}, found {len(runs)}")

run = runs[0]
config_dir = config_path.parent
census_path = (config_dir / run["hh_marg_file"]).resolve()
seed_path = (config_dir / run["hh_seed_data"]).resolve()
attributes = list(run["considered_atts"])
zone_field = run["zone_field"]
weight_column = run["pool_count_column"]

assert seed_path == (config_dir / run["hh_sample_file"]).resolve()
assert run["include_zero_cell_values"] is False

print("Census:   ", census_path)
print("Seed:     ", seed_path)
print("Controls: ", attributes)
print("Weight:   ", weight_column)

Census:    /Users/bobla/Documents/repos/PopSyn_Monash/IO/processed_data/census_POA_hh_saa.csv
Seed:      /Users/bobla/Documents/repos/PopSyn_Monash/IO/processed_data/ori_sample_hh.csv
Controls:  ['hhsize', 'totalvehs', 'dwelltype', 'owndwell', 'hhinc']
Weight:    _weight


In [3]:
marginals = pd.read_csv(census_path, header=[0, 1])
zone_column = next(column for column in marginals.columns if column[0] == zone_field)
marginals = marginals.set_index(zone_column)
marginals.index.name = zone_field
marginals = marginals.loc[:, marginals.columns.get_level_values(0).isin(attributes)]
marginals = marginals.apply(pd.to_numeric, errors="raise")

seed = pd.read_csv(seed_path, usecols=[*attributes, weight_column])
seed[weight_column] = pd.to_numeric(seed[weight_column], errors="raise")
if seed[attributes + [weight_column]].isna().any().any():
    raise ValueError("The IPF inputs contain missing values")
if (seed[weight_column] <= 0).any():
    raise ValueError(f"{weight_column} must contain positive weights")

# IPF needs one row per joint seed cell, with its survey weights summed.
seed_cells = (
    seed.groupby(attributes, as_index=False, dropna=False)[weight_column]
    .sum()
)

print(f"{len(seed):,} seed records condensed to {len(seed_cells):,} joint cells")
print(f"{len(marginals):,} Census zones loaded")

30,803 seed records condensed to 3,303 joint cells
691 Census zones loaded


## Resolve unsupported Census states

Classical IPF cannot create a joint cell absent from the seed. The only positive unsupported control in these files is `Negative income`; it is combined with the supported `Nil income` state. The checks below fail loudly if the input schemas later introduce another unsupported state.

In [4]:
structural_zero_folds = {
    ("hhinc", "Negative income"): "Nil income",
}
fold_log = []

for (attribute, unsupported_state), supported_state in structural_zero_folds.items():
    source = (attribute, unsupported_state)
    destination = (attribute, supported_state)
    seed_states = set(seed_cells[attribute])
    if source in marginals.columns and unsupported_state not in seed_states:
        if destination not in marginals.columns or supported_state not in seed_states:
            raise ValueError(f"Cannot fold {source} into unsupported destination {destination}")
        folded_total = int(marginals[source].sum())
        marginals[destination] = marginals[destination] + marginals[source]
        marginals = marginals.drop(columns=[source])
        fold_log.append({"source": source, "destination": destination, "count": folded_total})

unsupported = {}
for attribute in attributes:
    seed_states = set(seed_cells[attribute])
    unsupported[attribute] = [
        state for state in marginals[attribute].columns
        if state not in seed_states and marginals[(attribute, state)].sum() > 0
    ]
unsupported = {attribute: states for attribute, states in unsupported.items() if states}
if unsupported:
    raise ValueError(f"Positive Census states are absent from the seed: {unsupported}")

zone_totals = pd.DataFrame({attribute: marginals[attribute].sum(axis=1) for attribute in attributes})
if not zone_totals.nunique(axis=1).eq(1).all():
    bad_zones = zone_totals.index[~zone_totals.nunique(axis=1).eq(1)].tolist()[:10]
    raise ValueError(f"Control totals disagree across attributes for zones: {bad_zones}")

pd.DataFrame(fold_log)

,source,destination,count
0,"(hhinc, Negative income)","(hhinc, Nil income)",17134


## Fit each zone

`ipfn.ipfn(...).iteration()` is the complete fitting step. Integer counts use the repository's existing TRS implementation so every output zone retains its Census household total.

In [5]:
def maximum_marginal_error(frame, targets, value_column):
    errors = {}
    for attribute, target in zip(attributes, targets):
        achieved = frame.groupby(attribute, observed=True)[value_column].sum()
        errors[attribute] = float(achieved.sub(target, fill_value=0).abs().max())
    return errors

zones = marginals.index[:MAX_ZONES] if MAX_ZONES is not None else marginals.index
zone_results = []
diagnostics = []
started = time.perf_counter()
np.random.seed(RANDOM_SEED)

for position, zone_id in enumerate(zones, start=1):
    targets = [marginals.loc[zone_id, attribute].astype(float) for attribute in attributes]
    dimensions = [[attribute] for attribute in attributes]
    fitter = ipfn.ipfn(
        seed_cells.copy(),
        targets,
        dimensions,
        weight_col=weight_column,
        convergence_rate=CONVERGENCE_RATE,
        max_iteration=MAX_ITERATIONS,
        verbose=0,
    )
    fitted = fitter.iteration()
    household_total = int(zone_totals.loc[zone_id, attributes[0]])
    fractional_errors = maximum_marginal_error(fitted, targets, weight_column)
    fitted["count"] = TRS(fitted[weight_column].to_numpy(), household_total)
    integer_errors = maximum_marginal_error(fitted, targets, "count")
    fitted.insert(0, zone_field, zone_id)
    fitted = fitted.loc[fitted["count"] > 0, [zone_field, *attributes, weight_column, "count"]]
    zone_results.append(fitted)
    diagnostics.append({
        zone_field: zone_id,
        "households": household_total,
        "fractional_max_abs_error": max(fractional_errors.values()),
        "integer_max_abs_error": max(integer_errors.values()),
        **{f"fractional_error_{key}": value for key, value in fractional_errors.items()},
        **{f"integer_error_{key}": value for key, value in integer_errors.items()},
    })
    if position == 1 or position % 25 == 0 or position == len(zones):
        elapsed = time.perf_counter() - started
        print(f"Finished {position:,}/{len(zones):,} zones in {elapsed:,.1f}s")

ipf_result = pd.concat(zone_results, ignore_index=True)
diagnostics = pd.DataFrame(diagnostics)

ipfn converged: convergence_rate not updating or below rate_tolerance
Finished 1/691 zones in 0.8s
ipfn converged: convergence_rate not updating or below rate_tolerance
ipfn converged: convergence_rate not updating or below rate_tolerance
Finished 25/691 zones in 9.5s
ipfn converged: convergence_rate not updating or below rate_tolerance
Finished 50/691 zones in 14.4s
Finished 75/691 zones in 19.5s
Finished 100/691 zones in 24.7s
Finished 125/691 zones in 30.2s
Finished 150/691 zones in 35.1s
Finished 175/691 zones in 40.5s
Finished 200/691 zones in 45.6s
Finished 225/691 zones in 51.7s
ipfn converged: convergence_rate not updating or below rate_tolerance
Finished 250/691 zones in 60.0s
ipfn converged: convergence_rate not updating or below rate_tolerance
ipfn converged: convergence_rate not updating or below rate_tolerance
Finished 275/691 zones in 72.4s
Maximum iterations reached
Finished 300/691 zones in 85.5s
ipfn converged: convergence_rate not updating or below rate_tolerance
Fini

## Validate and save comparable outputs

In [6]:
expected_totals = zone_totals.loc[zones, attributes[0]].astype(np.int64).sort_index()
actual_totals = ipf_result.groupby(zone_field, observed=True)["count"].sum().astype(np.int64).sort_index()
pd.testing.assert_series_equal(actual_totals, expected_totals, check_names=False)

counts_path = output_dir / "household_counts.csv"
diagnostics_path = output_dir / "diagnostics.csv"
ipf_result[[zone_field, *attributes, "count"]].to_csv(counts_path, index=False)
diagnostics.to_csv(diagnostics_path, index=False)

if SAVE_FRACTIONAL_WEIGHTS:
    fractional_path = output_dir / "household_fractional_weights.csv"
    ipf_result[[zone_field, *attributes, weight_column]].to_csv(fractional_path, index=False)
else:
    fractional_path = None

print(f"Integer counts:     {counts_path}")
print(f"Fractional weights: {fractional_path}")
print(f"Diagnostics:        {diagnostics_path}")
diagnostics.describe()

Integer counts:     /Users/bobla/Documents/repos/PopSyn_Monash/IO/output/ipf_weighted_seed_full_two_stage_baseline/household_counts.csv
Fractional weights: /Users/bobla/Documents/repos/PopSyn_Monash/IO/output/ipf_weighted_seed_full_two_stage_baseline/household_fractional_weights.csv
Diagnostics:        /Users/bobla/Documents/repos/PopSyn_Monash/IO/output/ipf_weighted_seed_full_two_stage_baseline/diagnostics.csv


,zone_id,households,fractional_max_abs_error,integer_max_abs_error,fractional_error_hhsize,fractional_error_totalvehs,fractional_error_dwelltype,fractional_error_owndwell,fractional_error_hhinc,integer_error_hhsize,integer_error_totalvehs,integer_error_dwelltype,integer_error_owndwell,integer_error_hhinc
count,691.000000,691.000000,691.000000,691.000000,691.000000,691.000000,691.000000,691.000000,6.910000e+02,691.000000,691.000000,691.000000,691.000000,691.000000
mean,3470.991317,3502.490593,1.455332,19.150507,1.114655,1.015140,0.663336,0.365160,8.683068e-02,13.827786,13.535456,8.924747,12.672938,10.558611
std,297.733993,5052.244137,7.718751,10.102543,6.134286,6.158717,4.898806,3.255818,2.097891e+00,8.555573,8.634098,9.077452,8.646360,5.420356
min,3000.000000,55.000000,0.000039,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3197.500000,293.000000,0.000437,11.000000,0.000405,0.000227,0.000012,0.000110,1.776357e-15,7.000000,7.000000,3.000000,6.000000,6.000000
50%,3451.000000,1195.000000,0.001736,17.000000,0.001160,0.001106,0.000061,0.000769,7.105427e-15,12.000000,12.000000,6.000000,11.000000,10.000000
75%,3739.500000,5512.500000,0.007750,25.000000,0.005073,0.006223,0.000745,0.004096,5.684342e-14,18.000000,18.000000,12.000000,17.000000,14.000000
max,3996.000000,38768.000000,76.172464,71.000000,55.000000,57.984381,76.172464,55.000000,5.500000e+01,55.000000,58.000000,71.000000,55.000000,55.000000
